In [ ]:
from pathlib import Path
import pandas as pd

CLASSES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]


## 2.3a — Verify train/val/test split

- Counts per split × class (should match the split log).
- Verify each image has a matching label (zero orphans).
- Verify no image appears in more than one split (leakage check).


In [21]:
from collections import defaultdict

SPLIT_ROOT = Path(r"C:\Users\ycode\Desktop\2nd_YouCode\briefs\MRI_class_DL\data_set_split")
SPLITS_3 = ["train", "val", "test"]

# --- 1. counts per split/class ---
rows = []
for sp in SPLITS_3:
    for c in CLASSES:
        base = SPLIT_ROOT / sp / c
        n_img = len(list((base / "images").glob("*.jpg"))) if base.exists() else 0
        n_lbl = len(list((base / "labels").glob("*.txt"))) if base.exists() else 0
        rows.append({"split": sp, "class": c, "images": n_img, "labels": n_lbl,
                     "img_no_lbl": n_img - n_lbl})
df_counts = pd.DataFrame(rows)
print("=== Counts per (split, class) ===")
display(df_counts)
print("\nTotals per split:")
print(df_counts.groupby("split")[["images", "labels"]].sum())
print("\nTotals per class (all splits):")
print(df_counts.groupby("class")[["images", "labels"]].sum())

# --- 2. leakage: same (class, stem) appearing in >1 split ---
seen = defaultdict(set)
for sp in SPLITS_3:
    for c in CLASSES:
        for p in (SPLIT_ROOT / sp / c / "images").glob("*.jpg"):
            seen[(c, p.stem)].add(sp)

leaks = {k: v for k, v in seen.items() if len(v) > 1}
print(f"\n=== Leakage check (same class+stem in multiple splits): {len(leaks)} ===")
if leaks:
    for k, v in list(leaks.items())[:20]:
        print(f"  {k} -> appears in {sorted(v)}")
else:
    print("  None. Clean.")

# --- 3. quick per-split sanity: no empty class folders ---
print("\n=== Per-split class presence ===")
for sp in SPLITS_3:
    present = [c for c in CLASSES if (SPLIT_ROOT / sp / c / "images").exists()
                                     and any((SPLIT_ROOT / sp / c / "images").iterdir())]
    print(f"  {sp:6s}: {present}")

=== Counts per (split, class) ===


,split,class,images,labels,img_no_lbl
0,train,Glioma,979,979,0
1,train,Meningioma,1226,1226,0
2,train,No Tumor,604,604,0
3,train,Pituitary,1217,1217,0
4,val,Glioma,135,135,0
5,val,Meningioma,140,140,0
6,val,No Tumor,100,100,0
7,val,Pituitary,136,136,0
8,test,Glioma,173,173,0
9,test,Meningioma,216,216,0



Totals per split:
       images  labels
split                
test      711     711
train    4026    4026
val       511     511

Totals per class (all splits):
            images  labels
class                     
Glioma        1287    1287
Meningioma    1582    1582
No Tumor       811     811
Pituitary     1568    1568

=== Leakage check (same class+stem in multiple splits): 0 ===
  None. Clean.

=== Per-split class presence ===
  train : ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
  val   : ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
  test  : ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']


In [22]:
YOLO_ROOT = Path(r"C:\Users\ycode\Desktop\2nd_YouCode\briefs\MRI_class_DL\data_set_yolo")
rows = []
for sp in ["train","val","test"]:
    imgs = list((YOLO_ROOT/"images"/sp).glob("*.jpg"))
    lbls = list((YOLO_ROOT/"labels"/sp).glob("*.txt"))
    empty = sum(1 for p in lbls if not p.read_text().strip())
    rows.append({"split": sp, "images": len(imgs), "labels": len(lbls),
                 "empty_labels": empty, "non_empty": len(lbls)-empty})
display(pd.DataFrame(rows))
print((YOLO_ROOT/"data.yaml").read_text())

,split,images,labels,empty_labels,non_empty
0,train,4026,4026,604,3422
1,val,511,511,100,411
2,test,711,711,107,604


path: C:\Users\ycode\Desktop\2nd_YouCode\briefs\MRI_class_DL\data_set_yolo
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: Glioma
  1: Meningioma
  2: No Tumor
  3: Pituitary



In [23]:
YOLO_ROOT = Path(r"C:\Users\ycode\Desktop\2nd_YouCode\briefs\MRI_class_DL\data_set_yolo")

# Find empty label files whose filename DOESN'T start with "No_Tumor__"
odd_empty = []
for sp in ["train", "val", "test"]:
    for p in (YOLO_ROOT / "labels" / sp).glob("*.txt"):
        if p.read_text().strip() == "" and not p.name.startswith("No_Tumor__"):
            odd_empty.append((sp, p.name))

print(f"Empty labels from non-No-Tumor folders: {len(odd_empty)}")
for x in odd_empty:
    print(" ", x)

Empty labels from non-No-Tumor folders: 0


## 2.4 — Final audit

Full verification of both derived datasets before moving to modeling.
- Classification (`data_set_split/`): counts, no orphans, no leakage.
- Detection (`data_set_yolo/`): counts, empty labels only in No Tumor, no leakage.
- `data.yaml` sanity.

In [25]:
from collections import defaultdict

SPLIT_ROOT = Path(r"C:\Users\ycode\Desktop\2nd_YouCode\briefs\MRI_class_DL\data_set_split")
YOLO_ROOT  = Path(r"C:\Users\ycode\Desktop\2nd_YouCode\briefs\MRI_class_DL\data_set_yolo")
SPLITS_3   = ["train", "val", "test"]
CLASSES    = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]

problems = []

# ============ A. data_set_split — counts ============
print("=" * 60)
print("A. data_set_split/ — counts per (split, class)")
print("=" * 60)
EXPECTED_SPLIT = {
    ("train", "Glioma"): 979, ("train", "Meningioma"): 1226,
    ("train", "No Tumor"): 604, ("train", "Pituitary"): 1217,
    ("val",   "Glioma"): 135, ("val",   "Meningioma"): 140,
    ("val",   "No Tumor"): 100, ("val", "Pituitary"): 136,
    ("test",  "Glioma"): 173, ("test",  "Meningioma"): 216,
    ("test",  "No Tumor"): 107, ("test", "Pituitary"): 215,
}
for sp in SPLITS_3:
    for c in CLASSES:
        base = SPLIT_ROOT / sp / c
        n_img = len(list((base / "images").glob("*.jpg"))) if base.exists() else 0
        n_lbl = len(list((base / "labels").glob("*.txt"))) if base.exists() else 0
        exp = EXPECTED_SPLIT.get((sp, c))
        ok = (n_img == n_lbl == exp)
        marker = "✓" if ok else "✗"
        print(f"  {marker} {sp:6s} {c:12s} : {n_img:4d} img / {n_lbl:4d} lbl (expected {exp})")
        if not ok:
            problems.append(f"Split count mismatch: {sp}/{c} got {n_img}/{n_lbl}, expected {exp}")

# ============ B. data_set_split — leakage ============
print("\n" + "=" * 60)
print("B. data_set_split/ — leakage check")
print("=" * 60)
seen = defaultdict(set)
for sp in SPLITS_3:
    for c in CLASSES:
        for p in (SPLIT_ROOT / sp / c / "images").glob("*.jpg"):
            seen[(c, p.stem)].add(sp)
leaks = {k: v for k, v in seen.items() if len(v) > 1}
print(f"  Same (class, stem) in multiple splits: {len(leaks)}")
if leaks:
    for k, v in list(leaks.items())[:5]:
        print(f"    {k} -> {sorted(v)}")
    problems.append(f"{len(leaks)} leakage(s) in data_set_split")

# ============ C. data_set_yolo — counts + empty labels ============
print("\n" + "=" * 60)
print("C. data_set_yolo/ — counts + empty labels")
print("=" * 60)
EXPECTED_YOLO = {"train": 4026, "val": 511, "test": 711}
total_empty = 0
for sp in SPLITS_3:
    imgs = list((YOLO_ROOT / "images" / sp).glob("*.jpg"))
    lbls = list((YOLO_ROOT / "labels" / sp).glob("*.txt"))
    empties = [p for p in lbls if not p.read_text().strip()]
    total_empty += len(empties)
    ok = (len(imgs) == len(lbls) == EXPECTED_YOLO[sp])
    marker = "✓" if ok else "✗"
    print(f"  {marker} {sp:6s}: {len(imgs):4d} img / {len(lbls):4d} lbl / {len(empties):4d} empty (expected total {EXPECTED_YOLO[sp]})")
    if not ok:
        problems.append(f"YOLO count mismatch: {sp} got {len(imgs)}/{len(lbls)}")

print(f"\n  Total empty labels: {total_empty} (expected ≈ 809 from log)")

# ============ D. data_set_yolo — empty only in No Tumor ============
print("\n" + "=" * 60)
print("D. data_set_yolo/ — empties should ONLY be No Tumor images")
print("=" * 60)
odd_empty = []
for sp in SPLITS_3:
    for p in (YOLO_ROOT / "labels" / sp).glob("*.txt"):
        if not p.read_text().strip() and not p.name.startswith("No_Tumor__"):
            odd_empty.append((sp, p.name))
print(f"  Empty labels outside No_Tumor: {len(odd_empty)}")
if odd_empty:
    for x in odd_empty[:10]:
        print(f"    {x}")
    problems.append(f"{len(odd_empty)} unexpected empty labels in YOLO dataset")

# ============ E. data_set_yolo — leakage ============
print("\n" + "=" * 60)
print("E. data_set_yolo/ — leakage check")
print("=" * 60)
seen_y = defaultdict(set)
for sp in SPLITS_3:
    for p in (YOLO_ROOT / "images" / sp).glob("*.jpg"):
        seen_y[p.stem].add(sp)
leaks_y = {k: v for k, v in seen_y.items() if len(v) > 1}
print(f"  Same stem in multiple splits: {len(leaks_y)}")
if leaks_y:
    for k, v in list(leaks_y.items())[:5]:
        print(f"    {k} -> {sorted(v)}")
    problems.append(f"{len(leaks_y)} leakage(s) in data_set_yolo")

# ============ F. data.yaml sanity ============
print("\n" + "=" * 60)
print("F. data.yaml")
print("=" * 60)
yaml_path = YOLO_ROOT / "data.yaml"
print(yaml_path.read_text() if yaml_path.exists() else "MISSING")
if not yaml_path.exists():
    problems.append("data.yaml missing")

# ============ SUMMARY ============
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
if problems:
    print(f"❌ {len(problems)} problem(s) found:")
    for p in problems:
        print(f"   - {p}")
else:
    print("✅ ALL CHECKS PASSED — Step 2 is complete.")

A. data_set_split/ — counts per (split, class)
  ✓ train  Glioma       :  979 img /  979 lbl (expected 979)
  ✓ train  Meningioma   : 1226 img / 1226 lbl (expected 1226)
  ✓ train  No Tumor     :  604 img /  604 lbl (expected 604)
  ✓ train  Pituitary    : 1217 img / 1217 lbl (expected 1217)
  ✓ val    Glioma       :  135 img /  135 lbl (expected 135)
  ✓ val    Meningioma   :  140 img /  140 lbl (expected 140)
  ✓ val    No Tumor     :  100 img /  100 lbl (expected 100)
  ✓ val    Pituitary    :  136 img /  136 lbl (expected 136)
  ✓ test   Glioma       :  173 img /  173 lbl (expected 173)
  ✓ test   Meningioma   :  216 img /  216 lbl (expected 216)
  ✓ test   No Tumor     :  107 img /  107 lbl (expected 107)
  ✓ test   Pituitary    :  215 img /  215 lbl (expected 215)

B. data_set_split/ — leakage check
  Same (class, stem) in multiple splits: 0

C. data_set_yolo/ — counts + empty labels
  ✓ train : 4026 img / 4026 lbl /  604 empty (expected total 4026)
  ✓ val   :  511 img /  511 lb

## 2.5 — Step 2 findings

### Cleanup applied
- 14 misfiled images moved to correct class folders (verified visually).
- 1 in-place relabel: `m3 (120)` class 3 → class 1 (frontal meningioma).
- 2 orphan labels dropped (Val/No Tumor: `image(53)`, `image(60)`).
- 2 empty labels created (Val/No Tumor: `image(55)`, `image(61)`).
- 1 unlabeled tumor-folder image dropped (Val/Glioma: `gg (342)` — looks No Tumor).
- Rule added to `cleanup.py`: any tumor-folder image with empty label is dropped.

### Split (`data_set_split/`)
| split | Glioma | Meningioma | No Tumor | Pituitary | Total |
|---|---|---|---|---|---|
| train | 979 | 1226 | 604 | 1217 | 4026 |
| val   | 135 | 140 | 100 | 136 | 511 |
| test  | 173 | 216 | 107 | 215 | 711 |

Stratified 85/15 from Train; original Val preserved untouched. Zero leakage.

### YOLO dataset (`data_set_yolo/`)
- Flat layout: `images/{train,val,test}` + `labels/{train,val,test}`.
- 4,026 / 511 / 711 images — one-to-one with labels.
- No Tumor boxes stripped → 811 empty labels (all in No Tumor).
- Class-3 boxes stripped from Glioma/Meningioma → cleaner detection targets.
- `data.yaml` written (absolute path, `nc: 4`, classes 0–3).

### Final audit
- ✅ All counts match expected exactly.
- ✅ Zero orphans (img/lbl pairing intact).
- ✅ Zero leakage (no stem in multiple splits).
- ✅ Zero empty labels outside No Tumor.
- ✅ `data.yaml` valid.

### Ready for Step 3
- Classification: `data_set_split/` — 4 classes, 4026/511/711.
- Detection: `data_set_yolo/` — tumor boxes only, 4 classes, flat layout.
- Pipeline is **reproducible** from `data_set/` (untouched) via `cleanup.py → split.py → yolo_prep.py`.

## 3.6 — Preprocessing & Augmentation design (implemented in Step 3)

The datasets are built. Now we define how images are fed into the models.

### Classification (Keras, Step 3)
- **Resize**: 224×224 (MobileNetV2 native input size).
- **Normalize**: `tf.keras.applications.mobilenet_v2.preprocess_input` (scales to [-1, 1]).
- **Train augmentation** (applied on-the-fly, per batch):
  - Rotation: ±15°
  - Translation: ±10%
  - Zoom: ±15%
  - Contrast: ±15%
  - Horizontal flip: enabled (brain MRI is roughly symmetric).
- **Val/Test**: resize + normalize only, `shuffle=False`, no augmentation → deterministic.

### Detection (YOLOv8, Step 5)
- **Resize**: 640×640 (Ultralytics default `imgsz`).
- **Train augmentation**: Ultralytics defaults (mosaic, flips, HSV, scale, translate) — bounding boxes are warped internally by the library.
- **Val/Test**: `imgsz=640`, no augmentation.

### Notes
- The brief's "adapter les boîtes englobantes aux transformations géométriques"
  is handled by **Ultralytics automatically** during YOLO training — no manual
  bbox transformation script needed.
- For classification, bounding boxes are irrelevant (Keras reads the folder name),
  so augmentation only needs to affect the image, not any labels.